In [ ]:
import re
import json
from pathlib import Path

# -------- File-specific naming --------
input_file = "informative_text.txt"
output_file = "chapter7_informativeText2.json"

chapter7_structured_data = {
    "chapter": None,
    "name": None,
    "intro": "",
    "sections": {}
}

# -------- Load text --------
with open(input_file, "r", encoding="utf-8") as f:
    chapter7_lines = f.readlines()

# -------- Regex patterns --------
chapter_pattern = re.compile(r"^Chapter\s+(\d+)", re.IGNORECASE)
section_pattern = re.compile(r"^(\d+\.\d+)\s+(.*)")
subsection_pattern = re.compile(r"^(\d+\.\d+\.\d+)\s+(.*)")

current_section = None
current_subsection = None

i = 0
while i < len(chapter7_lines):
    line = chapter7_lines[i].strip()

    if not line:
        i += 1
        continue

    # ---- Chapter ----
    chapter_match = chapter_pattern.match(line)
    if chapter_match:
        chapter7_structured_data["chapter"] = chapter_match.group(1)

        #  Get chapter name from next non-empty line
        j = i + 1
        while j < len(chapter7_lines) and not chapter7_lines[j].strip():
            j += 1

        if j < len(chapter7_lines):
            chapter7_structured_data["name"] = chapter7_lines[j].strip()
            i = j  # move pointer forward

        i += 1
        continue

    # ---- Subsection ----
    subsection_match = subsection_pattern.match(line)
    if subsection_match:
        subsection_id = subsection_match.group(1)
        subsection_title = subsection_match.group(2)

        if current_section:
            chapter7_structured_data["sections"][current_section]["subsections"][subsection_id] = {
                "title": subsection_title,
                "content": ""
            }
            current_subsection = subsection_id
        i += 1
        continue

    # ---- Section ----
    section_match = section_pattern.match(line)
    if section_match:
        section_id = section_match.group(1)
        section_title = section_match.group(2)

        chapter7_structured_data["sections"][section_id] = {
            "title": section_title,
            "content": "",
            "subsections": {}
        }

        current_section = section_id
        current_subsection = None
        i += 1
        continue

    # ---- Content routing ----
    if current_subsection:
        chapter7_structured_data["sections"][current_section]["subsections"][current_subsection]["content"] += line + " "
    elif current_section:
        chapter7_structured_data["sections"][current_section]["content"] += line + " "
    else:
        chapter7_structured_data["intro"] += line + " "

    i += 1

# -------- Save JSON --------
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(chapter7_structured_data, f, indent=2, ensure_ascii=False)

print("Structured JSON created successfully.")

Structured JSON created successfully.


*Problem:*
Headings and titles are not clearly distinguishable in the text file, causing poor segmentation during parsing or chunking.

*Solution:*
Use consistent, explicit structural markers (e.g., ## Heading) instead of relying on formatting like uppercase text, ensuring reliable separation and hierarchy detection.